# Problem description - Flow matching for precipitation downscaling

# Description of the dataset (WRF, 4km resolution, maximum 3 day snowfall, based on GFDL, CFSR, CCSM 1981-2060) and what the dataloader does (divides data into blocks, decimates to simulate low-res climate forcing, augments with high-res topography)

In [ ]:
import torch
import numpy as np
import os
import rasterio
from notebook_utils import block_mean_tensor_vectorized

class WRFDataLoader:
    def __init__(self, data_dir='./data',swe_factor=100.0,hgt_factor=300.0):
        self.data_dir = data_dir
        self.swe_factor = swe_factor
        self.hgt_factor = hgt_factor
    
        # Ensure data directory exists
        if not os.path.exists(self.data_dir):
            print(f"Creating data directory: {self.data_dir}")
            os.makedirs(self.data_dir)
    
        rasters = []
        filtered_files = []
        for f in os.listdir(data_dir):
            if '.tif' in f and 'HGT' not in f and 'aux' not in f:
                filtered_files.append(f)
        for f in filtered_files:
            try:
                data = rasterio.open(f'{data_dir}/{f}')
            except rasterio.RasterioIOError:
                continue
            snow = data.read().squeeze()[1:-1,1:-1]
            rasters.append(snow)
    
        self.rasters = np.stack(rasters,axis=0)
        self.rasters[self.rasters==-999] = np.nan
        self.rasters /= swe_factor
    
        self.rasters_test = self.rasters[120:]
        self.rasters = self.rasters[:120]
    
        self.d,self.h,self.w = self.rasters.shape
        self.d_test, _, _ = self.rasters_test.shape
    
        hgt = rasterio.open(f'{data_dir}/geo_southeast_4km_HGT.tif').read().squeeze()
        self.hgt = hgt[1:-1,1:-1] / hgt_factor
    
    def sample(self, n, n_pix=64, device='cpu',test_set=False):
    
        if test_set:
            d = self.d_test
        else:
            d = self.d
    
        images = []
        heights = []
        crses = []
        while len(images) < n:
          index = np.random.randint(d)
          h0 = np.random.randint(self.h - n_pix)
          w0 = np.random.randint(self.w - n_pix)
          if test_set:
              img = torch.tensor(self.rasters_test[index,h0:h0+n_pix,w0:w0+n_pix])
          else:
              img = torch.tensor(self.rasters[index,h0:h0+n_pix,w0:w0+n_pix])
          if torch.isnan(img).sum() < 0.3*n_pix**2:
              images.append(img)
              heights.append(torch.tensor(self.hgt[h0:h0+n_pix,w0:w0+n_pix]))
    
        image_batch = torch.stack(images).reshape(-1,1,n_pix,n_pix)
        hgt_batch = torch.stack(heights).reshape(-1,1,n_pix,n_pix)
        crs_batch = block_mean_tensor_vectorized(image_batch,4)
    
        return image_batch,hgt_batch,crs_batch
    
    def pull(self, locs, n_pix=64, device='cpu',test_set=False):
    
        if test_set:
            d = self.d_test
        else:
            d = self.d
    
        images = []
        heights = []
        crses = []
        for index,h0,w0 in locs:
          if test_set:
              img = torch.tensor(self.rasters_test[index,h0:h0+n_pix,w0:w0+n_pix])
          else:
              img = torch.tensor(self.rasters[index,h0:h0+n_pix,w0:w0+n_pix])
          if torch.isnan(img).sum() < 0.3*n_pix**2:
              images.append(img)
              heights.append(torch.tensor(self.hgt[h0:h0+n_pix,w0:w0+n_pix]))
    
        image_batch = torch.stack(images).reshape(-1,1,n_pix,n_pix)
        hgt_batch = torch.stack(heights).reshape(-1,1,n_pix,n_pix)
        crs_batch = block_mean_tensor_vectorized(image_batch,4)
    
        return image_batch,hgt_batch,crs_batch

n_pix = 64
target_model = WRFDataLoader(data_dir='./data/sx3/')
device = 'cuda'



In [ ]:
import matplotlib.pyplot as plt

def plot_blended_hillshade(z_sample,ax,cmap=plt.cm.terrain,ve=1,dx=4000,vmin=-2.5,vmax=2.5):
   ls = LightSource(azdeg=315, altdeg=45)
   rgb = ls.shade(z_sample.numpy(), cmap=cmap, 
                  blend_mode='overlay', vert_exag=ve, 
                  dx=dx, dy=dx,vmin=vmin,vmax=vmax)
   ax.imshow(rgb)
    
hgt_factor = target_model.hgt_factor
swe_factor = target_model.swe_factor
x,h,c = target_model.sample(1,n_pix=n_pix)

fig,axs = plt.subplots(ncols=3)
for ax in axs.ravel():
    plot_blended_hillshade(h.cpu().squeeze()*swe_factor,ax,cmap=plt.cm.grey,vmin=0,vmax=3000,ve=100)
    ax.xaxis.set_visible(False)
    ax.yaxis.set_visible(False)

c_hgt = axs[0].imshow(h.squeeze()*hgt_factor,cmap=plt.cm.gist_earth,alpha=0.8)
_     = axs[1].imshow(x.squeeze()*swe_factor,cmap=plt.cm.plasma,vmin=0.0,vmax=400.0,alpha=0.8)
c_swe = axs[2].imshow(c.squeeze()*swe_factor,cmap=plt.cm.plasma,vmin=0.0,vmax=400.0,alpha=0.8)

cax = inset_axes(axs[0], width="5%", height="50%", loc='lower left',bbox_to_anchor=(-0.15, 0.15, 1, 1), bbox_transform=axs[0].transAxes)
cbar = plt.colorbar(c_hgt,cax=cax,orientation='vertical')
cbar.ax.tick_params(rotation=45)
cbar.ax.yaxis.set_ticks_position('left')
cbar.ax.yaxis.set_label_position('left')
cbar.set_label('Elevation (m)')

cax = inset_axes(axs[2], width="5%", height="50%", loc='lower left',bbox_to_anchor=(1.0, 0.15, 1, 1), bbox_transform=axs[2].transAxes)
cbar = plt.colorbar(c_swe,cax=cax,orientation='vertical')
cbar.ax.tick_params(rotation=45)
cbar.set_label('Snowfall (mm)')

fig.subplots_adjust(wspace=0,hspace=0)
fig.set_size_inches(12,4)


# Define the flow matching objective and the relevant tools

In [ ]:
class OTProbabilityPaths:
    def __init__(self,sigma_0):
        self.sigma_0 = sigma_0

    def get_mu(self,x1,t):
        return x1*t

    def get_sigma(self,t):
        return 1 - (1 - self.sigma_0)*t

    def get_psi(self,x,x1,t):
        return self.get_sigma(t)*x + self.get_mu(x1,t)

    def get_v_target(self,x,x1,t):
        return x1 - (1-self.sigma_0)*x0

fm = OTProbabilityPaths(1e-3)

# Define the model for approximating the velocity (here a u-net augmented with self-attention)

In [ ]:
from models.unet_models import UNet

in_channels = 4

v_model = UNet(
        in_channels=in_channels,
        model_channels=64,
        out_channels=1,
        num_res_blocks=2,
        attention_resolutions=(16,),  # Self-attention at 16x16 resolution
        channel_mult=(1, 1, 2, 3, 4,),   # Four resolution levels: 64, 32, 16, 8, 4
        dropout=0.1,
    ).to(device)

# Perform the training loop

In [ ]:
TRAIN = True
if TRAIN:
    optimizer = torch.optim.Adam(v_model.parameters(),lr=2e-4)
    
    batch_size = 24
    n_steps = 10000
    v_model.train()
    for i in range(n_steps):
        # Clear gradient buffer
        optimizer.zero_grad()

        # Sample from data points and associated conditioning
        x1,h,c = (f.to(device) for f in target_model.sample(batch_size,n_pix=n_pix))

        # mask any coarse resolution data
        c[c.isnan()] = 0.0

        # Build mask for unresolved high-res data - this will be a signal to ignore
        m = x1.isnan()  

        # Eliminate any nans
        x1[m] = 0.0

        # Sample time in [0,1]
        t = torch.rand(x0.shape[0],1,1,1,device=device)
        
        # Sample noise
        x0 = torch.randn_like(x1,device=device)

        # Extrapolate denoised image
        xt = fm.get_psi(x0,x1,t)

        # Build conditions - 
        # partially denoised image, topography, coarse climate, mask
        x_cat = torch.cat((xt,h,c,m),dim=1)

        # Predict velocity
        v_pred = v_model(x_cat,t.squeeze())

        # Get conditional velocity
        v_target = fm.get_v_target(x0,x1,t)

        # Compute loss, backprop, step
        loss = ((v_pred - v_target)**2)[~m].mean()
        loss.backward()
        optimizer.step()
        
        if i%100==0:
            print(i,loss.item())
            torch.save(f'./checkpoints/conditional_snowfall_model_latest.pt',weights_only=False)
            torch.save(f'./checkpoints/conditional_snowfall_model_{i:04d}.pt',weights_only=False)
else:
    v_model = torch.load('./checkpoints/conditional_snowfall_model_latest.pt',weights_only=False)

# Generate some samples via ODE integration.  Applied to test set low-res climate for comparison.

In [ ]:
from torchdiffeq import odeint

# Number of samples to generate
n_samples = 3

# Draw some test set samples from a few tiles
xtrue,h,c = (f.to(device) for f in 
               target_model.pull([(29,0,0),(11,20,100),(5,50,220)],
               test_set=True))

# Mask coarse conditioning with nans
c[c.isnan()] = 0.0

# Set velocity model to eval mode
v_model.eval()

# Sample some noise
x0 = torch.randn(n_samples,1,n_pix,n_pix,device=device)

# Define the right-hand side for the flow-matching ODE
def func(t,x):
    xin = torch.cat((x,h,c,torch.zeros_like(x)),dim=1)
    return v_model(xin,t.reshape(1))

# Define integration interval
t = torch.linspace(0,1,10,device=device)

# Generate new samples via integration of the flow-matching ODE
# From noise initial conditions along trajectories defined
# by the velocity model
with torch.no_grad():
    trajectories = odeint(func, x0, t,rtol=1e-5,atol=1e-5)

# Finished samples occur at final time step
xsamp = trajectories[-1]


# Visualize in comparison to ground-truth WRF runs.  

In [ ]:
from matplotlib.colors import LightSource
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

fig,axs = plt.subplots(nrows=4,ncols=n_samples)
for i in range(n_samples):
    plot_blended_hillshade(h[i].cpu().squeeze()*hgt_factor,axs[0,i],cmap=plt.cm.grey,vmin=0,vmax=3000,ve=100)
    plot_blended_hillshade(h[i].cpu().squeeze()*hgt_factor,axs[1,i],cmap=plt.cm.grey,vmin=0,vmax=3000,ve=100)
    plot_blended_hillshade(h[i].cpu().squeeze()*hgt_factor,axs[3,i],cmap=plt.cm.grey,vmin=0,vmax=3000,ve=100)
    plot_blended_hillshade(h[i].cpu().squeeze()*hgt_factor,axs[2,i],cmap=plt.cm.grey,vmin=0,vmax=3000,ve=100)

    cswe = axs[0,i].imshow(xtrue[i].cpu().squeeze()*swe_factor,vmin=0,vmax=400,cmap=plt.cm.plasma,alpha=0.7)
    _    = axs[1,i].imshow(xsamp[i].cpu().squeeze()*swe_factor,vmin=0,vmax=400,cmap=plt.cm.plasma,alpha=0.7)
    _    = axs[2,i].imshow(    c[i].cpu().squeeze()*swe_factor,vmin=0,vmax=400,cmap=plt.cm.plasma,alpha=0.7)
    chgt = axs[3,i].imshow(    h[i].cpu().squeeze()*hgt_factor,cmap=plt.cm.gist_earth,vmin=-500,vmax=3000,alpha=0.7)


    if i==n_samples-1:
        cax = inset_axes(axs[0,i], width="5%", height="50%", loc='lower left',bbox_to_anchor=(1.0, 0.2, 1, 1), bbox_transform=axs[0,i].transAxes)
        cbar = plt.colorbar(cswe,cax=cax,orientation='vertical')
        cbar.ax.tick_params(rotation=45)
        cbar.set_label('Snowfall (mm)')

        cax = inset_axes(axs[3,i], width="5%", height="50%", loc='lower left',bbox_to_anchor=(1.0, 0.2, 1, 1), bbox_transform=axs[3,i].transAxes)
        cbar = plt.colorbar(chgt,cax=cax,orientation='vertical')
        cbar.ax.tick_params(rotation=45)
        cbar.set_label('Elevation (m)')

for ax in axs.ravel():
    ax.xaxis.set_visible(False)
    ax.yaxis.set_visible(False)

axs[0,0].set_title('WRF', rotation='vertical',x=-0.1,y=0.3)
axs[1,0].set_title('Flow matching', rotation='vertical',x=-0.1,y=0.3)
axs[2,0].set_title('Low-res', rotation='vertical',x=-0.1,y=0.3)
axs[3,0].set_title('Elevation', rotation='vertical',x=-0.1,y=0.3)

fig.set_size_inches(9*3/4.,9)
fig.subplots_adjust(wspace=0,hspace=0)
#fig.savefig('figures/conditional_snowfall_sample.pdf',bbox_inches='tight')

# Demonstrate the variability in predicted samples

In [ ]:
n_samples = 50
xtrue,h,c = (f.to(device) for f in 
               target_model.pull([(11,20,100)],
               test_set=True))

c[c.isnan()] = 0.0

xtrue = torch.tile(xtrue,(n_samples,1,1,1))
h = torch.tile(h,(n_samples,1,1,1))
c = torch.tile(c,(n_samples,1,1,1))

v_model.eval()

x0 = torch.randn(n_samples,1,n_pix,n_pix,device=device)

def func(t,x):
    xin = torch.cat((x,h,c,torch.zeros_like(x)),dim=1)
    return v_model(xin,t.reshape(1))

t = torch.linspace(0,1,10,device=device)
with torch.no_grad():
    trajectories = odeint(func, x0, t,rtol=1e-5,atol=1e-5)

xsamp = trajectories[-1]
xmean = xsamp.mean(dim=0)
xdelta = xsamp - xmean

fig,axs = plt.subplots(nrows=1,ncols=5)
axs = axs.ravel()
letters = 'abcde'
lc = 0

for ax in axs.ravel():
    plot_blended_hillshade(h[i].cpu().squeeze()*swe_factor,ax,cmap=plt.cm.grey,vmin=0,vmax=3000,ve=100)
    ax.xaxis.set_visible(False)
    ax.yaxis.set_visible(False)
    

for i in range(3):    
    c_delta = axs[i].imshow(xdelta[i].cpu().squeeze()*swe_factor,vmin=-100,vmax=100,alpha=0.7,cmap=plt.cm.coolwarm)

    axs[i].text(0.05, 0.9, letters[lc],color='white',transform=axs[i].transAxes,fontsize=12)
    lc+=1

cax = inset_axes(axs[0], width="5%", height="50%", loc='lower left',bbox_to_anchor=(-0.15, 0.15, 1, 1), bbox_transform=axs[0].transAxes)
cbar = plt.colorbar(c_delta,cax=cax,orientation='vertical')
cbar.ax.tick_params(rotation=45)#,color='white', labelcolor='white')
cbar.ax.yaxis.set_ticks_position('left')
cbar.ax.yaxis.set_label_position('left')
cbar.set_label('$\\Delta$ Snowfall (mm)')

axs[3].imshow(c[0].cpu().squeeze()*swe_factor,vmin=0,vmax=400,alpha=0.7,cmap=plt.cm.plasma)
axs[3].text(0.05, 0.9, letters[4],color='white',transform=axs[3].transAxes,fontsize=12)

c_abs = axs[4].imshow(xmean.cpu().squeeze()*swe_factor,vmin=0,vmax=400,alpha=0.7,cmap=plt.cm.plasma)
cax = inset_axes(axs[4], width="5%", height="50%", loc='lower left',bbox_to_anchor=(1.0, 0.15, 1, 1), bbox_transform=axs[4].transAxes)
cbar = plt.colorbar(c_abs,cax=cax,orientation='vertical')
cbar.ax.tick_params(rotation=45)
cbar.set_label('Snowfall (mm)')

axs[4].text(0.05, 0.9, letters[4],color='white',transform=axs[4].transAxes,fontsize=12)

fig.set_size_inches(12,6)
fig.subplots_adjust(wspace=0,hspace=0)

# Compute summary statistics

In [ ]:
fig,axs = plt.subplots(nrows=1,ncols=2)
for ax in axs.ravel():
    plot_blended_hillshade(h[i].cpu().squeeze()*swe_factor,ax,cmap=plt.cm.grey,vmin=0,vmax=3000,ve=100)
    ax.xaxis.set_visible(False)
    ax.yaxis.set_visible(False)

c_mean = axs[0].imshow(xsamp.mean(axis=0).squeeze().cpu()*swe_factor,vmin=0,vmax=400,alpha=0.8,cmap=plt.cm.plasma)
c_std  = axs[1].imshow(xsamp.std(axis=0).squeeze().cpu()*swe_factor,vmin=0,vmax=100,alpha=0.8)

cax = inset_axes(axs[0], width="5%", height="50%", loc='lower left',bbox_to_anchor=(-0.15, 0.15, 1, 1), bbox_transform=axs[0].transAxes)
cbar = plt.colorbar(c_mean,cax=cax,orientation='vertical')
cbar.ax.tick_params(rotation=45)
cbar.ax.yaxis.set_ticks_position('left')
cbar.ax.yaxis.set_label_position('left')
cbar.set_label('mean Snowfall (mm)')

cax = inset_axes(axs[1], width="5%", height="50%", loc='lower left',bbox_to_anchor=(1.0, 0.15, 1, 1), bbox_transform=axs[1].transAxes)
cbar = plt.colorbar(c_std,cax=cax,orientation='vertical')
cbar.ax.tick_params(rotation=45)
cbar.set_label('std Snowfall (mm)')


fig.set_size_inches(12,6)
fig.subplots_adjust(wspace=0,hspace=0)

# Compute CRPS

In [2]:
import scores